In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
# Inclusive min/max nuclei volume used to filter predicted labels
# Set to None to skip label-size filtering
MIN_MAX_NUCLEI_VOLUME = (150,2000)

# Path containing your batch processing results
BP_CSV_RESULTS_PATH = "./results/bp_results/WT_organoids/per_label_results_WT_organoids_MIN_None_MAX_None_CT1_NP1.csv"

In [ ]:
# Read Dataframe with batch processing results from disk (.csv copy)
final_df = pd.read_csv(BP_CSV_RESULTS_PATH)
final_df.head()

In [ ]:
# Filter out rows where volume_nuclei is less than MIN or greater than MAX (based on visual inspection and volume distribution)
final_df_filtered = final_df[(final_df["area_nuclei"] >= MIN_MAX_NUCLEI_VOLUME[0]) & (final_df["area_nuclei"] <= MIN_MAX_NUCLEI_VOLUME[1])]

In [ ]:
# Group by organoid identifiers and compute per-organoid averages
grouped_df = final_df_filtered.groupby(["experiment_id", "mouse_id", "treatment_id", "replica_id"]).agg({
    "intensity_mean_nuclei": "mean",
    "area_nuclei": "mean",
    "intensity_mean_cyto": "mean",
    "area_cyto": "mean",
    "nuclei_cyto_ratio": "mean",
}).reset_index()

# Save average results
if MIN_MAX_NUCLEI_VOLUME is not None:
    average_path = bp_results_dir / f"average_nuclear_cyto_intensity_{input_folder_id}_MIN_{MIN_MAX_NUCLEI_VOLUME[0]}_MAX{MIN_MAX_NUCLEI_VOLUME[1]}_CT{CYTOPLASM_THICKNESS}.csv"
else:
    average_path = bp_results_dir / f"average_nuclear_cyto_intensity_{input_folder_id}_MIN_None_MAX_None_CT{CYTOPLASM_THICKNESS}.csv"
grouped_df.to_csv(average_path, index=False)
average_path

In [ ]:
# Discard in memory Dataframe and read from disk (.csv copy)
grouped_df = pd.read_csv(average_path)
grouped_df.head()

In [ ]:
# Create the box plot
fig = px.box(
    grouped_df,
    x="treatment_id",
    y="intensity_mean_nuclei",
    color="treatment_id",
    points="all",
    hover_data=["experiment_id", "mouse_id", "replica_id"],
    title="YAP Nuclear Intensity Average by Treatment ID",
)

fig.show()

# Create the box plot
fig = px.box(
    grouped_df,
    x="treatment_id",
    y="intensity_mean_cyto",
    color="treatment_id",
    points="all",
    hover_data=["experiment_id", "mouse_id", "replica_id"],
    title="YAP Cytoplasmic Intensity Average by Treatment ID",
)

fig.show()

# Create the box plot
fig = px.box(
    grouped_df,
    x="treatment_id",
    y="nuclei_cyto_ratio",
    color="treatment_id",
    points="all",
    hover_data=["experiment_id", "mouse_id", "replica_id"],
    title="YAP Nuclear/Cytoplasmic signal Ratio by Treatment ID",
)

fig.show()

In [ ]:
# Group by organoid identifiers and compute per-organoid averages
grouped_bio_rep_df = grouped_df.groupby(["mouse_id", "treatment_id"]).agg({
    "intensity_mean_nuclei": "mean",
    "area_nuclei": "mean",
    "intensity_mean_cyto": "mean",
    "area_cyto": "mean",
    "nuclei_cyto_ratio": "mean",
}).reset_index()

# Drop any row where 'treatment_id' contains the string "dmPGE2"
grouped_bio_rep_df = grouped_bio_rep_df[~grouped_bio_rep_df["treatment_id"].str.contains("dmPGE2", na=False)]
grouped_bio_rep_df

In [ ]:
import plotly.graph_objects as go

# Define the custom order for treatment_id
treatment_order = ["BCM", "MSN", "PGE2", "BCM_60min", "MSN_60min", "PGE2_60min"]

# Calculate means for each treatment in order
means = (
    grouped_bio_rep_df
    .set_index('treatment_id')
    .loc[treatment_order]  # preserve order
    .reset_index()
    [["treatment_id", "nuclei_cyto_ratio"]]
    .groupby("treatment_id")["nuclei_cyto_ratio"]
    .mean()
    .reindex(treatment_order)
    .values
)

# Scatter plot for individual data points, enforcing the order on x
fig = go.Figure()

for treatment in treatment_order:
    sub_df = grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == treatment]
    fig.add_trace(
        go.Scatter(
            x=[treatment] * len(sub_df),
            y=sub_df["nuclei_cyto_ratio"],
            mode="markers",
            name=treatment,
            text=sub_df["mouse_id"],
            marker=dict(size=10),
            showlegend=False  # disables individual duplicate legend entries
        )
    )

# Add mean as individual (not connected) diamond points for each treatment
for treatment, mean in zip(treatment_order, means):
    fig.add_trace(
        go.Scatter(
            x=[treatment],
            y=[mean],
            mode="markers",
            name="Mean" if treatment == treatment_order[0] else None,
            marker=dict(
                symbol="line-ew",
                size=28,
                color="black",
                line=dict(width=3, color="black"),
            ),

            showlegend=(treatment == treatment_order[0]),  # Only show legend entry once
        )
    )

fig.update_layout(
    xaxis=dict(title="treatment_id", categoryorder="array", categoryarray=treatment_order),
    yaxis_title="nuclei_cyto_ratio",
    title="YAP Nuclear/Cytoplasmic signal Ratio by Treatment ID (Mean ± Individual Points)",
)

fig.show()

In [ ]:
import scipy.stats as stats

# Define comparison groups
ttest_comparisons = [
    ("MSN", "BCM"),
    ("PGE2", "BCM"),
    ("MSN_60min", "BCM_60min"),
    ("PGE2_60min", "BCM_60min"),
]

anova_groups = [
    (["BCM", "MSN", "PGE2"], "BCM"),
    (["BCM_60min", "MSN_60min", "PGE2_60min"], "BCM_60min"),
]

print("==== T-TEST COMPARISONS BETWEEN GROUPS (nuclei_cyto_ratio) ====")
for case, ctrl in ttest_comparisons:
    case_vals = grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == case]["nuclei_cyto_ratio"]
    ctrl_vals = grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == ctrl]["nuclei_cyto_ratio"]
    tstat, pval = stats.ttest_ind(case_vals, ctrl_vals, equal_var=False, nan_policy='omit')
    print(f"{case} vs {ctrl}: t={tstat:.3f}, p={pval:.4g}")

print("\n==== 1-WAY ANOVA AND PAIRWISE COMPARISONS (nuclei_cyto_ratio) ====")
for groups, control in anova_groups:
    # Collect data for all groups
    group_vals = [grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == g]["nuclei_cyto_ratio"].dropna() for g in groups]
    anova_stat, anova_p = stats.f_oneway(*group_vals)
    print(f"ANOVA for {groups}: F={anova_stat:.3f}, p={anova_p:.4g}")
    # Pairwise t-test for each group vs control
    for grp in groups:
        if grp == control:
            continue
        grp_vals = grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == grp]["nuclei_cyto_ratio"]
        ctrl_vals = grouped_bio_rep_df[grouped_bio_rep_df["treatment_id"] == control]["nuclei_cyto_ratio"]
        tstat, pval = stats.ttest_ind(grp_vals, ctrl_vals, equal_var=False, nan_policy='omit')
        print(f"  {grp} vs {control}: t={tstat:.3f}, p={pval:.4g}")